In [1]:
import torch

In [2]:
#exetute this cell only in Google Colab
!pip install transformers

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 4.2 MB 25.6 MB/s 
     |████████████████████████████████| 86 kB 6.4 MB/s 
     |████████████████████████████████| 596 kB 66.0 MB/s 
     |████████████████████████████████| 6.6 MB 62.5 MB/s 
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 3.13
    Uninstalling PyYAML-3.13:
      Successfully uninstalled PyYAML-3.13


In [3]:
from transformers import pipeline
import os
import torch
import numpy as np
import pandas as pd
import pickle

In [4]:
#exetute this cell only in Google Colab
from google.colab import files
import io

In [5]:
#get index for the maximum score from prediction in model distilbert-base-uncased-emotion
############################################################
def get_max_index(ld):
    lv = [d['score'] for d in ld]
    lvind = lv.index(max(lv))
    return lvind
############################################################

In [6]:
#get label with the maximum score for model distilbert-base-uncased-mnli
############################################################
def get_max_label(ld):
    val = ld['scores']
    val_ind = lv.index(max(lv))
    label = ld['labels'][val_ind]
    return label
############################################################

In [7]:
# If there's a GPU available...
if torch.cuda.is_available():    
    # Tell PyTorch to use the GPU.    
    device = torch.device("cuda")
    print('There are %d GPU(s) available.' % torch.cuda.device_count())
    print('We will use the GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

There are 1 GPU(s) available.
We will use the GPU: Tesla T4


In [8]:
#exetute this cell only in Google Colab
uploaded = files.upload()
rmp = pd.read_csv(io.BytesIO(uploaded['rmp_small.csv']))

Saving rmp_small.csv to rmp_small.csv


In [9]:
rmp

,review,score
0,"he can be a mean man sometimes, but just get o...",2
1,"He's fun, but he speeds through lectures and l...",2
2,Dr. Foster is an amazing teacher. One of my fa...,4
3,She is one of the greatest teacher I have ever...,4
4,Professor Bird was very easy and relaxed. If y...,3
...,...,...
97540,Charismatic and enthusiastic but that's it . H...,0
97541,"Great books - Dickens, Eliot, etc. A little d...",3
97542,Awesome professor. Doesnt stop explaining unti...,4
97543,"He has a really high standard, but grades fair...",4


In [10]:
sentences = list(rmp['review'])
labels = list(rmp['score'])

In [11]:
#remove nan from both lists
selnan = list()
for idx,name in enumerate(sentences):
    if name == '' or pd.isnull(name):
        selnan.append(idx)
sentences = [sentences[i] for i, e in enumerate(sentences) if i not in selnan]
labels = [labels[i] for i, e in enumerate(labels) if i not in selnan]
(len(sentences), len(labels))

(97545, 97545)

In [12]:
#do zero-shot classification
#check documentation https://huggingface.co/typeform/distilbert-base-uncased-mnli
classifier = pipeline("zero-shot-classification", 
                      model = "Narsil/deberta-large-mnli-zero-cls",
                      device = 0 
)

Downloading:   0%|          | 0.00/729 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.51G [00:00<?, ?B/s]

Some weights of the model checkpoint at Narsil/deberta-large-mnli-zero-cls were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Downloading:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/878k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/446k [00:00<?, ?B/s]

In [13]:
l1 = "show up to class room on time or never misses class"
l2 = "strong knowledge and solid education or patient and caring"
l3 = "encourages the students to question"
l4 = "using real world example or exercises real world problems"
l5 = "clear or sufficient syllabus"
l6 = "course have useful topics or updated topics"
candidate_labels=[l1, l2, l3, l4, l5, l6]

In [14]:
prediction = classifier(sentences, candidate_labels=candidate_labels)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [15]:
prediction[90], prediction[101]

({'labels': ['using real world example or exercises real world problems',
   'encourages the students to question',
   'strong knowledge and solid education or patient and caring',
   'clear or sufficient syllabus',
   'course have useful topics or updated topics',
   'show up to class room on time or never misses class'],
  'scores': [0.24168947339057922,
   0.22851410508155823,
   0.17598138749599457,
   0.16666759550571442,
   0.1526125818490982,
   0.03453483432531357],
  'sequence': 'Frustrating. Most of the work was filler, and there wasn\'t much substance to the class. Some people call it "weeding-out" uninterested folk, I call it lack of organization.'},
 {'labels': ['clear or sufficient syllabus',
   'encourages the students to question',
   'using real world example or exercises real world problems',
   'course have useful topics or updated topics',
   'strong knowledge and solid education or patient and caring',
   'show up to class room on time or never misses class'],
  's

In [16]:
lv = prediction[90]['scores']
lv

[0.24168947339057922,
 0.22851410508155823,
 0.17598138749599457,
 0.16666759550571442,
 0.1526125818490982,
 0.03453483432531357]

In [17]:
lvind = lv.index(max(lv))
lvind

0

In [18]:
l_max_labels = [get_max_label(val) for val in prediction]

In [19]:
df = pd.DataFrame(list(zip(l_max_labels, labels)), columns =['prof_opinion', 'score'])
df

,prof_opinion,score
0,using real world example or exercises real wor...,2
1,using real world example or exercises real wor...,2
2,strong knowledge and solid education or patien...,4
3,strong knowledge and solid education or patien...,4
4,course have useful topics or updated topics,3
...,...,...
97540,using real world example or exercises real wor...,0
97541,course have useful topics or updated topics,3
97542,strong knowledge and solid education or patien...,4
97543,using real world example or exercises real wor...,4


In [20]:
#calculate crosstabulations
result = pd.crosstab(df["prof_opinion"], df["score"])
result

score,0,1,2,3,4
prof_opinion,,,,,
clear or sufficient syllabus,686,702,1358,3031,4361
course have useful topics or updated topics,724,913,1692,3332,4343
encourages the students to question,3577,2431,2422,3101,3115
show up to class room on time or never misses class,76,75,101,128,145
strong knowledge and solid education or patient and caring,102,351,1773,9184,19103
using real world example or exercises real world problems,10826,5898,4863,4574,4558


In [ ]:
data = pd.read_csv('output.csv')

In [ ]:
data.drop("Unnamed: 0",inplace=True,axis=1)

In [ ]:
data

In [ ]:
prof_1_data = data.query("prof_opinion == 'strong knowledge and solid education or patient and caring' | prof_opinion == 'encourages the students to question' | prof_opinion == 'show up to class room on time or never misses class' ")

In [ ]:
cor_1_data = data.query("prof_opinion == 'course have useful topics or updated topics' | prof_opinion == 'clear or sufficient syllabus' | prof_opinion == 'using real world example or exercises real world problems' ")

In [ ]:
pd.crosstab(cor_1_data['prof_opinion'],cor_1_data['score']).plot(kind="barh",figsize=(8,6),width = 0.8)
plt.grid(True)
plt.title('lebel regarding course \n (distilbart-mnli-12-1 model)')
plt.xlabel('score')
plt.xlim(0,11000)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig('crosstab_cor_1_data.pdf',dpi = 300)

In [ ]:
pd.crosstab(prof_1_data['prof_opinion'],prof_1_data['score']).plot(kind="barh",figsize=(8,6),width = 0.8)
plt.grid(True)
plt.title('label regarding Professor \n (distilbart-mnli-12-1 model)')
plt.xlabel('score')
plt.tight_layout()
plt.savefig('crosstab_prof_1_data.pdf',dpi = 300)

In [ ]:
data2 = pd.read_csv('output2.csv')

In [ ]:
data2.drop("Unnamed: 0",inplace=True,axis=1)

In [ ]:
prof_2_data = data2.query("prof_opinion == 'strong knowledge and solid education or patient and caring' | prof_opinion == 'encourages the students to question' | prof_opinion == 'show up to class room on time or never misses class' ")

In [ ]:
cor_2_data = data2.query("prof_opinion == 'course have useful topics or updated topics' | prof_opinion == 'clear or sufficient syllabus' | prof_opinion == 'using real world example or exercises real world problems' ")

In [ ]:
pd.crosstab(cor_2_data['prof_opinion'],cor_2_data['score']).plot(kind="barh",figsize=(8,6),width = 0.8)
plt.grid(True)
plt.title('label regarding course \n (deberta-large-mnli-zero-cls model)')
plt.xlabel('score')
plt.tight_layout()
plt.savefig('crosstab_cor_2_data.pdf',dpi = 300)

In [ ]:
pd.crosstab(prof_2_data['prof_opinion'],prof_2_data['score']).plot(kind="barh",figsize=(8,6),width = 0.8)
plt.grid(True)
plt.title('label regarding Professor \n (deberta-large-mnli-zero-cls model)')
plt.xlabel('score')
plt.legend(loc="lower right")
plt.xlim(0,29000)
plt.tight_layout()
plt.savefig('crosstab_prof_2_data.pdf',dpi = 300)